# Akan-BPE — Gemma-3-1B × Akan TTS tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-gemma-1b.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-gemma-1b.ipynb)

QLoRA fine-tune of a base LLM with the Akan TTS tokenizer, scored against the unmodified base on **bits-per-byte (BPB, full byte coverage)**. Runs two arms — random and mean-of-subword embedding init — and compares them. Run top-to-bottom on a Kaggle/Colab **T4 GPU** (~45–50 min per arm).

> **Gated model.** Accept the license for `google/gemma-3-1b-pt` on Hugging Face and authenticate in the login cell before running.

## Setup

In [1]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 650, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 650 (delta 83), reused 99 (delta 59), pack-reused 516 (from 1)
Receiving objects: 100% (650/650), 1.11 MiB | 17.51 MiB/s, done.
Resolving deltas: 100% (407/407), done.
/kaggle/working/akan-bpe
Working directory: /kaggle/working/akan-bpe


In [2]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.6 MB/s eta 0:00:00


In [3]:
# Hugging Face authentication. Paste a token with access to the gated models
# and datasets used below when prompted (input is hidden).
from getpass import getpass

from huggingface_hub import login

login()

In [4]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [5]:
# Download Akan datasets. --tts-limit 50000 pins the 45,000/2,500/2,500 split.
!python scripts/download.py --output-dir data --tts-limit 50000

README.md: 29.2kB [00:00, 50.0MB/s]
Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 30450.71it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 1.64MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [6]:
# Train the TTS tokenizer if not already present
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    print("TTS tokenizer already present, skipping.")

TTS tokenizer already present, skipping.


In [7]:
# Verify required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

All inputs ready.


## Run — random vs mean-of-subword embedding init

Defaults (eval cap, max-length, LoRA rank, learning rate, etc.) live in `scripts/model_integration.py`; pass `--help` to see or override them. Each arm writes its own adapter dir + result JSON derived from the model id.

In [8]:
# Arm A — random embedding init (default)
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt

Building eval dataset: 100%|██████████████| 512/512 [00:00<00:00, 1480.86text/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/google/gemma-3-1b-pt/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 419, in cached_files
    hf_hub_download(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    return fn

In [9]:
# Arm B — mean-of-subword embedding init
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt --embedding-init-mode mean_subword

Building eval dataset: 100%|██████████████| 512/512 [00:00<00:00, 1456.14text/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/google/gemma-3-1b-pt/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 419, in cached_files
    hf_hub_download(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    return fn

## Results — compare the two arms

In [10]:
import json
from pathlib import Path

def load_eval(path):
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    b = r["eval"]["bpb"]
    return {
        "init": r["embedding_init_mode"],
        "eval_loss": r["eval"]["eval_loss"],
        "perplexity": r["eval"]["perplexity"],
        "exp_bpb": b["experiment"]["bits_per_byte"],
        "base_bpb": (b["base"] or {}).get("bits_per_byte"),
        "improvement": b.get("improvement"),
    }

arms = {
    "random": load_eval("results/run-gemma-1b.json"),
    "mean_subword": load_eval("results/run-gemma-1b-meansub.json"),
}
header = f"{'metric':<24}{'random':>14}{'mean_subword':>16}"
print(header)
print("-" * len(header))

def show(label, key, fmt="{:.4f}"):
    cells = [fmt.format(arms[a][key]) if arms[a][key] is not None else "—"
             for a in ("random", "mean_subword")]
    print(f"{label:<24}{cells[0]:>14}{cells[1]:>16}")

show("eval_loss", "eval_loss")
show("perplexity", "perplexity", "{:.2f}")
show("experiment BPB", "exp_bpb")
show("base BPB", "base_bpb")
show("improvement (base−exp)", "improvement")

best = min(arms, key=lambda a: arms[a]["exp_bpb"])
print(f"\nLower fine-tuned BPB: {best!r} arm.")

FileNotFoundError: [Errno 2] No such file or directory: 'results/run-gemma-1b.json'